# Sesión 3 · Práctica guiada: vuestras funciones para leer y descargar datos

Esta práctica la hacemos juntos, paso a paso: el profesor escribe y explica cada celda, y tú la reproduces en tu notebook.

Dos partes: primero leeréis y limpiaréis un CSV con el formato del INE; después convertiréis en tabla una respuesta de su API.

In [ ]:
# Preparacion: localiza la carpeta de la asignatura (y la descarga si estas en Google Colab).
# Ejecuta esta celda primero; no hace falta entenderla todavia.
import os, subprocess
from pathlib import Path

REPO = "https://github.com/luisaugustos/ppy-analisis-social.git"


def _buscar_carpeta():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        for candidata in (carpeta, carpeta / "ppy-analisis-social"):
            if (candidata / "datos" / "ORIGEN.md").is_file():
                return candidata
    return None


raiz = _buscar_carpeta()
if raiz is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    raiz = Path("ppy-analisis-social").resolve()

os.chdir(raiz)
print("Carpeta de trabajo lista. Los datos estan en la subcarpeta 'datos'.")

In [ ]:
import json
from pathlib import Path

import pandas as pd
import requests

## Parte A · Un CSV tal como viene

### Ejercicio 1 · Mirar antes de leer

Mostrad las cuatro primeras líneas de `datos/ine_hipotecas_bruto.csv` como texto (está en codificación `latin-1`). ¿Qué separador usa? ¿Cómo se escriben los miles?

In [ ]:
ruta = Path("datos/ine_hipotecas_bruto.csv")

# TODO 1
...

### Ejercicio 2 · Leerlo bien

Leed el fichero con `pd.read_csv` indicando separador, miles, valores perdidos (`..`) y codificación. La columna `Total` debe quedar como número.

In [ ]:
# TODO 2
...

assert bruto["Total"].dtype != object and bruto["Total"].max() > 1000
bruto.head(3)

### Ejercicio 3 · Código de provincia, nombre y fecha

Añadid a `bruto` tres columnas:

- `cod_prov`: los dos primeros caracteres de `Provincias` (y `"00"` para el total nacional);
- `provincia`: el nombre, sin el código;
- `fecha`: la fecha correspondiente a `Periodo` (formato `"%YM%m"`).

In [ ]:
# TODO 3
...

assert bruto["cod_prov"].nunique() == 53 and "Salamanca" in set(bruto["provincia"])
bruto[["Provincias", "cod_prov", "provincia", "Periodo", "fecha"]].tail(3)

### Ejercicio 4 · Quedarse con el número de hipotecas

Cread la tabla `numero` solo con las filas cuyo `Concepto` es `"Número de hipotecas"` y las columnas `cod_prov`, `provincia`, `fecha` y `Total`, renombrando `Total` como `hipotecas`.

In [ ]:
# TODO 4
...

assert list(numero.columns) == ["cod_prov", "provincia", "fecha", "hipotecas"]
numero.head(3)

### Ejercicio 5 · Vuestra función

Juntad los ejercicios 2, 3 y 4 en una función `leer_numero_hipotecas(ruta)`. Después usadla para obtener el total anual de hipotecas de Granada.

In [ ]:
# TODO 5
...


tabla = leer_numero_hipotecas("datos/ine_hipotecas_bruto.csv")
granada = tabla[tabla["provincia"] == "Granada"].set_index("fecha")["hipotecas"]
granada.resample("YS").sum()

## Parte B · La API del INE

La celda siguiente pide a la API los tres últimos meses de la tabla 3200. Si no hay conexión, usa una respuesta de ejemplo con la misma estructura.

In [ ]:
url = "https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/3200?tip=AM&nult=3"

try:
    respuesta = requests.get(url, timeout=30)
    respuesta.raise_for_status()
    series = respuesta.json()
    print("Respuesta recibida del INE")
except requests.RequestException:
    series = json.loads(Path("datos/ine_respuesta_ejemplo.json").read_text(encoding="utf-8"))
    print("Sin conexión: uso la respuesta de ejemplo")

### Ejercicio 6 · Explorar la respuesta

Averiguad con código: cuántas series hay, qué claves tiene la primera, cómo se llama y cuál es su primer dato.

In [ ]:
# TODO 6
...

### Ejercicio 7 · Del JSON a una tabla

Construid una lista `filas` con un diccionario por cada dato, con las claves `nombre`, `fecha` (los diez primeros caracteres de `Fecha`) y `valor`. Convertidla en la tabla `tabla_api`.

In [ ]:
# TODO 7
...

assert {"nombre", "fecha", "valor"} <= set(tabla_api.columns) and len(tabla_api) >= len(series)
tabla_api.head()

### Ejercicio 8 · Filtrar por texto

Quedaos con las filas cuyo `nombre` contiene `"Número de hipotecas"` y `"Salamanca"`. Pista: `tabla_api["nombre"].str.contains("...")`.

In [ ]:
# TODO 8
...

salamanca_api

### Ejercicio 9 · Guardar copia y reutilizarla

Escribid `guardar_copia(tabla, nombre)`, que guarde una tabla en `datos/descargas/<nombre>.csv` (creando la carpeta si no existe) y devuelva la ruta. Después comprobad que podéis volver a leerla.

In [ ]:
# TODO 9
...


ruta_copia = guardar_copia(tabla_api, "ine_3200_ultimos")
releida = pd.read_csv(ruta_copia, parse_dates=["fecha"])
assert len(releida) == len(tabla_api)
print(ruta_copia)

## Para cerrar

- ¿Qué ventajas tiene la API frente a descargar el CSV a mano? ¿Y algún inconveniente?
- Buscad en la web del INE otra tabla que os interese y anotad su identificador (aparece en «Descargar → Json»): la usaréis en el **entregable 1**.